# BT4012: Student-selected direct LabelSpreading comparison

**11 October 2026.** The student selected a new semi-supervised mechanism after the two autoencoder comparisons: direct LabelSpreading with 15 nearest neighbours, alpha 0.2, at most 30 iterations, tolerance 0.001 and two CPU threads. Keep all 165 original features, training-fitted standardization, clipping to ±10, sparse graph storage and the four chronological checks. Fit graphs only on eligible training rows; future/checking/test features do not fit them.

Compare newly fitted original XGBoost, known-only LabelSpreading, and LabelSpreading using both known and unknown rows. This fixed comparison performs no automatic parameter search, blend selection or Kaggle upload. The best recorded public score remains 0.94265; local results cannot guarantee a higher hidden score.

Only the supplied competition CSVs are required. The edge file is not used by this chosen mechanism: its graph represents **feature similarity**, not Bitcoin transfers. General algorithm references: [scikit-learn semi-supervised learning](https://scikit-learn.org/stable/modules/semi_supervised.html) and [LabelSpreading](https://scikit-learn.org/stable/modules/generated/sklearn.semi_supervised.LabelSpreading.html). No public competition solution, external labels or external data are used.

## 1. What the experiment tests

Think of each transaction as a point described by its 165 feature values. Standardization puts features on comparable scales. Each fitted point connects to its 15 nearest fitted points. LabelSpreading starts with confirmed labels and repeatedly spreads their information through this sparse similarity graph. Alpha 0.2 controls the graph contribution versus the fixed known-label signal; it does not mean that 20% of unknown rows receive labels. At most 30 rounds are allowed.

Unknown rows start with label **−1** and contribute their feature positions and connections. Their learned soft class distributions are model estimates, not confirmed answers. We do not convert unknown rows to licit or train another classifier on their inferred answers. The known-only control uses the same algorithm and settings while omitting unknown rows from its scaler and graph. This comparison tests the whole addition of unknown rows, including its effect on scaling and neighbours; it does not isolate a single causal mechanism.

For a future query, sum the fitted distributions of its 15 closest training nodes and normalize. If those nodes have zero total label information, use the confirmed training illicit share as the chosen fallback. The graph remains fixed during prediction. The future query does not become a fitted graph node and queries cannot pass information to each other.

In [1]:
from pathlib import Path
import gc
import hashlib
import json
import platform
import sys
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn import config_context
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import LabelSpreading
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier

data_locations = [Path.cwd() / 'Data Set', Path.cwd().parent / 'Data Set',
                  Path('/Users/mlvlo/BT4012/Kaggle/Data Set')]
DATA_DIR = next((p for p in data_locations if (p / 'Train Data.csv').is_file()), None)
assert DATA_DIR is not None, 'Set DATA_DIR to the supplied competition files.'
OUTPUT_DIR = Path.cwd() / 'labelspreading_results'
OUTPUT_DIR.mkdir(exist_ok=True)
FEATURES = [f'feat_{i}' for i in range(1, 166)]
CHECKS = [(14, 15, 28), (17, 18, 31), (19, 20, 33), (21, 22, 35)]
FAMILIES = ['original_xgboost', 'ls_known_only', 'ls_all_rows']
CPU_THREADS, DISTANCE_MEMORY_MIB, QUERY_BATCH = 2, 128, 2048
SEED, MAXIMUM_ROUNDS, STOPPING_PATIENCE = 42, 400, 60
LS_PARAMETERS = dict(kernel='knn', n_neighbors=15, alpha=0.2,
                     max_iter=30, tol=0.001, n_jobs=CPU_THREADS)
assert 'torch' not in sys.modules, 'Run in a fresh process/kernel.'

train = pd.read_csv(DATA_DIR / 'Train Data.csv', usecols=['txId', 'time_step', 'label'] + FEATURES)
assert train.txId.is_unique and train.time_step.between(1, 35).all()
assert np.isfinite(train[FEATURES].to_numpy()).all()
known = train.loc[train.label.notna()].copy()
assert known.label.isin([0, 1]).all()
known['label'] = known.label.astype(int)
assert known.label.nunique() == 2

def progress(**details):
    temporary = OUTPUT_DIR / 'progress.tmp'
    temporary.write_text(json.dumps(details))
    temporary.replace(OUTPUT_DIR / 'progress.json')

print('Training rows:', len(train), '; confirmed:', len(known),
      '; unknown:', int(train.label.isna().sum()))
print('scikit-learn:', sklearn.__version__, '; XGBoost:', xgboost.__version__)
print('Fixed LabelSpreading settings:', LS_PARAMETERS)
print('Two CPU threads; sparse k-neighbour graph; 128 MiB distance working-memory setting.')


Training rows: 141772 ; confirmed: 31235 ; unknown: 110537
scikit-learn: 1.9.1 ; XGBoost: 3.2.0
Fixed LabelSpreading settings: {'kernel': 'knn', 'n_neighbors': 15, 'alpha': 0.2, 'max_iter': 30, 'tol': 0.001, 'n_jobs': 2}
Two CPU threads; sparse k-neighbour graph; 128 MiB distance working-memory setting.


## 2. Fit only earlier information

Use training cutoffs 14, 17, 19 and 21 and check the following fourteen periods. For LabelSpreading, fit the scaler and graph on all eligible rows through the cutoff. Its settings are fixed; internal convergence stopping does not inspect checking AUC. No inner stopping split is needed for this fixed algorithm.

Retain the original XGBoost control's existing protocol: fit through cutoff minus three, choose tree count on the last three training periods, then refit confirmed labels through the cutoff. XGBoost keeps 15 leaves, 127 bins, learning rate 0.05, lambda 10, minimum child weight 3, full sampling and class weight 1; at most 400 trees with 60-round stopping. This preserves the comparator rather than retuning it.

These four checks overlap and have been reused for development. They give temporal stress tests but are not four independent holdout tests. Matching the future fourteen-period horizon makes them more relevant to this task than a random split. Other validation designs exist; this experiment retains the design selected by the student.

The implementation uses the standard LabelSpreading nearest-neighbour prediction calculation, with the selected prior fallback for zero-information queries. It verifies agreement with the library on a sample of queries with positive information. Library convergence warnings are recorded; they do not trigger a change in the selected iteration limit. The 128 MiB setting limits distance computation chunks, not total process memory.

In [2]:
def make_xgboost(rounds, *, stopping=False):
    return XGBClassifier(
        objective='binary:logistic', n_estimators=int(rounds), learning_rate=0.05,
        tree_method='hist', device='cpu', grow_policy='lossguide', max_depth=0,
        max_leaves=15, max_bin=127, min_child_weight=3, reg_lambda=10,
        subsample=1.0, colsample_bytree=1.0, scale_pos_weight=1.0,
        eval_metric='auc', early_stopping_rounds=STOPPING_PATIENCE if stopping else None,
        n_jobs=CPU_THREADS, random_state=SEED,
    )


def standardized(scaler, rows):
    values = np.asarray(rows[FEATURES], dtype=np.float32)
    return np.ascontiguousarray(np.clip(scaler.transform(values), -10, 10), dtype=np.float32)


def fit_pipeline(family, end):
    learning = known.loc[known.time_step <= end]
    if family == 'original_xgboost':
        fitting = learning.loc[learning.time_step <= end - 3]
        stopping = learning.loc[learning.time_step > end - 3]
        assert fitting.time_step.max() < stopping.time_step.min()
        assert fitting.label.nunique() == stopping.label.nunique() == 2
        progress(phase='XGBoost stopping', family=family, cutoff=end)
        with threadpool_limits(limits=CPU_THREADS):
            initial = make_xgboost(MAXIMUM_ROUNDS, stopping=True)
            initial.fit(fitting[FEATURES], fitting.label,
                        eval_set=[(stopping[FEATURES], stopping.label)], verbose=False)
            rounds = int(initial.best_iteration + 1)
            assert 1 <= rounds <= MAXIMUM_ROUNDS
            assert abs(roc_auc_score(stopping.label, initial.predict_proba(stopping[FEATURES])[:, 1])
                       - float(initial.best_score)) < 1e-12
        curve = initial.evals_result()['validation_0']['auc']
        pd.DataFrame({'round': np.arange(1, len(curve) + 1), 'inner_roc_auc': curve}).to_csv(
            OUTPUT_DIR / f'stopping_curve_original_xgboost_{end}.csv', index=False)
        progress(phase='XGBoost refitting', family=family, cutoff=end, rounds=rounds)
        with threadpool_limits(limits=CPU_THREADS):
            model = make_xgboost(rounds)
            model.fit(learning[FEATURES], learning.label)
        return dict(family=family, cutoff=end, model=model, scaler=None, rounds=rounds,
                    fitting_rows=len(learning), known_rows=len(learning), unknown_rows=0,
                    n_iter=None, convergence_warning=False, zero_mass_fitted_rows=0)

    assert family in ['ls_known_only', 'ls_all_rows']
    rows = (learning if family == 'ls_known_only' else train.loc[train.time_step <= end])
    assert rows.time_step.max() <= end
    labels = rows.label.fillna(-1).to_numpy(dtype=np.int64)
    assert np.isin(labels, [-1, 0, 1]).all()
    assert (labels != -1).sum() == len(learning)
    assert (labels == -1).sum() == int(rows.label.isna().sum())
    scaler = StandardScaler()
    with threadpool_limits(limits=CPU_THREADS):
        scaler.fit(np.asarray(rows[FEATURES], dtype=np.float32))
        features = standardized(scaler, rows)
    assert int(scaler.n_samples_seen_) == len(rows)
    model = LabelSpreading(**LS_PARAMETERS)
    progress(phase='LabelSpreading graph construction and fitting', family=family,
             cutoff=end, fitting_rows=len(rows), unknown_rows=int((labels == -1).sum()))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        with threadpool_limits(limits=CPU_THREADS), config_context(working_memory=DISTANCE_MEMORY_MIB):
            model.fit(features, labels)
    unexpected = [w for w in caught if not issubclass(w.category, ConvergenceWarning)]
    assert not unexpected, [(w.category.__name__, str(w.message)) for w in unexpected]
    assert np.array_equal(model.classes_, [0, 1])
    assert model.X_.shape == (len(rows), len(FEATURES))
    assert np.isfinite(model.label_distributions_).all()
    assert 1 <= model.n_iter_ <= LS_PARAMETERS['max_iter']
    masses = model.label_distributions_.sum(axis=1)
    assert np.allclose(masses[masses > 0], 1, rtol=0, atol=1e-12)
    assert masses[labels != -1].min() > 0
    return dict(family=family, cutoff=end, model=model, scaler=scaler, rounds=None,
                fitting_rows=len(rows), known_rows=len(learning), unknown_rows=int((labels == -1).sum()),
                prior=float(learning.label.mean()), n_iter=int(model.n_iter_),
                convergence_warning=bool(caught), warning_messages=[str(w.message) for w in caught],
                zero_mass_fitted_rows=int((masses == 0).sum()), features=FEATURES,
                fitting_txIds=rows.txId.to_numpy())


def predict_pipeline(pipeline, rows, *, audit_library=False):
    if pipeline['family'] == 'original_xgboost':
        with threadpool_limits(limits=CPU_THREADS):
            return pipeline['model'].predict_proba(rows[FEATURES])[:, 1], 0
    assert rows.time_step.min() > pipeline['cutoff']
    assert set(rows.txId).isdisjoint(pipeline['fitting_txIds'])
    model = pipeline['model']
    features = standardized(pipeline['scaler'], rows)
    probabilities = np.empty(len(rows), dtype=np.float64)
    zero_information = np.zeros(len(rows), dtype=bool)
    with threadpool_limits(limits=CPU_THREADS), config_context(working_memory=DISTANCE_MEMORY_MIB):
        for first in range(0, len(rows), QUERY_BATCH):
            last = min(first + QUERY_BATCH, len(rows))
            neighbours = model.nn_fit.kneighbors(features[first:last], return_distance=False)
            totals = model.label_distributions_[neighbours].sum(axis=1)
            masses = totals.sum(axis=1)
            informed = masses > 0
            batch = np.full(last - first, pipeline['prior'], dtype=np.float64)
            batch[informed] = totals[informed, 1] / masses[informed]
            probabilities[first:last] = batch
            zero_information[first:last] = ~informed
        if audit_library:
            selected = np.flatnonzero(~zero_information)[:64]
            if len(selected):
                standard = model.predict_proba(features[selected])[:, 1]
                assert np.allclose(standard, probabilities[selected], rtol=0, atol=1e-12)
    assert np.isfinite(probabilities).all()
    assert np.all((probabilities >= 0) & (probabilities <= 1))
    assert np.all(probabilities[zero_information] == pipeline['prior'])
    return probabilities, int(zero_information.sum())


## 3. Train the chosen comparison

Each condition predicts the same confirmed future transactions. Save transaction identifiers, true checking labels, periods, probabilities, training counts, convergence and fallback counts. No setting changes after looking at these outer scores.

In [3]:
records, saved = [], {}
started_all = time.perf_counter()
for end, first, last in CHECKS:
    checking = known.loc[known.time_step.between(first, last)]
    assert set(train.loc[train.time_step <= end].txId).isdisjoint(checking.txId)
    assert checking.label.nunique() == 2
    saved[f'labels_{end}'] = checking.label.to_numpy()
    saved[f'txIds_{end}'] = checking.txId.to_numpy()
    saved[f'time_steps_{end}'] = checking.time_step.to_numpy()
    for family in FAMILIES:
        started = time.perf_counter()
        pipeline = fit_pipeline(family, end)
        progress(phase='predicting chronological check', family=family, cutoff=end)
        probabilities, fallbacks = predict_pipeline(pipeline, checking, audit_library=True)
        auc = float(roc_auc_score(checking.label, probabilities))
        saved[f'{family}_{end}'] = probabilities
        row = dict(model=family, train_end=end, validation_start=first, validation_end=last,
                   fitting_rows=pipeline['fitting_rows'], known_rows=pipeline['known_rows'],
                   unknown_rows=pipeline['unknown_rows'], checking_rows=len(checking),
                   rounds=pipeline['rounds'], n_iter=pipeline['n_iter'],
                   convergence_warning=pipeline['convergence_warning'],
                   zero_mass_fitted_rows=pipeline['zero_mass_fitted_rows'],
                   fallback_queries=fallbacks, auc=auc, seconds=round(time.perf_counter() - started, 3))
        records.append(row)
        print(f'{family}: fit through {end}; check {first}–{last}; AUC={auc:.6f}; '
              f'unknown fitting rows={pipeline["unknown_rows"]:,}; fallback queries={fallbacks:,}', flush=True)
        pd.DataFrame(records).to_csv(OUTPUT_DIR / 'validation_results.csv', index=False)
        np.savez_compressed(OUTPUT_DIR / 'validation_probabilities.npz', **saved)
        del pipeline
        gc.collect()
    progress(phase='validation cutoff complete', cutoff=end)

results = pd.DataFrame(records)
comparison = results.pivot(index='train_end', columns='model', values='auc')[FAMILIES]
comparison['all_rows_minus_original'] = comparison.ls_all_rows - comparison.original_xgboost
comparison['all_rows_minus_known_only'] = comparison.ls_all_rows - comparison.ls_known_only
summary = comparison[FAMILIES].agg(['mean', 'min']).T
comparison.to_csv(OUTPUT_DIR / 'comparison.csv')
summary.to_csv(OUTPUT_DIR / 'summary.csv')
print('Four-check elapsed seconds:', round(time.perf_counter() - started_all, 2))
display(comparison.round(6))
display(summary.round(6))


original_xgboost: fit through 14; check 15–28; AUC=0.968521; unknown fitting rows=0; fallback queries=0
ls_known_only: fit through 14; check 15–28; AUC=0.921993; unknown fitting rows=0; fallback queries=0
ls_all_rows: fit through 14; check 15–28; AUC=0.903163; unknown fitting rows=56,094; fallback queries=5
original_xgboost: fit through 17; check 18–31; AUC=0.925611; unknown fitting rows=0; fallback queries=0
ls_known_only: fit through 17; check 18–31; AUC=0.861706; unknown fitting rows=0; fallback queries=0
ls_all_rows: fit through 17; check 18–31; AUC=0.874425; unknown fitting rows=64,134; fallback queries=17
original_xgboost: fit through 19; check 20–33; AUC=0.901582; unknown fitting rows=0; fallback queries=0
ls_known_only: fit through 19; check 20–33; AUC=0.850331; unknown fitting rows=0; fallback queries=0
ls_all_rows: fit through 19; check 20–33; AUC=0.828829; unknown fitting rows=68,482; fallback queries=5
original_xgboost: fit through 21; check 22–35; AUC=0.915330; unknown fit

model,original_xgboost,ls_known_only,ls_all_rows,all_rows_minus_original,all_rows_minus_known_only
train_end,,,,,
14,0.968521,0.921993,0.903163,-0.065359,-0.018830
17,0.925611,0.861706,0.874425,-0.051187,0.012718
19,0.901582,0.850331,0.828829,-0.072753,-0.021502
21,0.915330,0.854200,0.834371,-0.080959,-0.019829


,mean,min
model,,
original_xgboost,0.927761,0.901582
ls_known_only,0.872057,0.850331
ls_all_rows,0.860197,0.828829


## 4. Inspect periods without tuning on them

The pooled fourteen-period AUC is the selected main comparison. Per-period AUC describes where performance differs; it is undefined for a period with only one class. Do not combine duplicated transactions across overlapping checks into a supposedly independent test score.

In [4]:
step_records = []
for end, first, last in CHECKS:
    labels, steps = saved[f'labels_{end}'], saved[f'time_steps_{end}']
    for step in np.unique(steps):
        selected = steps == step
        row = dict(train_end=end, validation_step=int(step), known_labels=int(selected.sum()),
                   illicit=int(labels[selected].sum()))
        for family in FAMILIES:
            row[family] = (float(roc_auc_score(labels[selected], saved[f'{family}_{end}'][selected]))
                           if len(np.unique(labels[selected])) == 2 else np.nan)
        row['all_rows_minus_original'] = row['ls_all_rows'] - row['original_xgboost']
        step_records.append(row)
per_step = pd.DataFrame(step_records)
per_step.to_csv(OUTPUT_DIR / 'per_time_step.csv', index=False)
display(per_step.loc[per_step.train_end == 21].round(6))


,train_end,validation_step,known_labels,illicit,original_xgboost,ls_known_only,ls_all_rows,all_rows_minus_original
42,21,22,1763,158,0.998833,0.982487,0.978867,-0.019965
43,21,23,1187,53,0.957622,0.874788,0.865770,-0.091852
44,21,24,1126,137,0.795488,0.697150,0.674382,-0.121106
45,21,25,594,118,0.999484,0.965007,0.966565,-0.032919
46,21,26,517,96,0.946779,0.898815,0.869111,-0.077667
47,21,27,206,24,0.999542,0.983745,0.983745,-0.015797
48,21,28,284,85,1.000000,0.993970,0.996630,-0.003370
49,21,29,1174,329,0.775619,0.698169,0.688570,-0.087049
50,21,30,524,83,0.987323,0.966710,0.961424,-0.025900
51,21,31,710,106,0.772820,0.715958,0.650537,-0.122282


## 5. Full-training experimental artifacts

Refit both selected LabelSpreading conditions through step 35. The all-row graph and scaler use all 141,772 training transactions, including 110,537 unknowns. Only 31,235 confirmed answers initialize known labels. The known-only graph and scaler use only those 31,235 transactions. Test features are query inputs only.

Save each fitted scaler/model bundle and an experimental prediction CSV aligned to the supplied sample submission. Check that reloaded components reproduce exactly the same scores. Preparing these files does not choose either for submission.

In [5]:
test = pd.read_csv(DATA_DIR / 'Test Data.csv')
template = pd.read_csv(DATA_DIR / 'Sample Submission' / 'BT4012 Competition Submission.csv')
assert test.txId.is_unique and test['index'].is_unique and template['index'].is_unique
assert len(test) == len(template) and set(test['index']) == set(template['index'])
assert test.time_step.between(36, 49).all() and train.time_step.max() < test.time_step.min()
assert set(train.txId).isdisjoint(test.txId)
assert np.isfinite(test[FEATURES].to_numpy()).all()
final_records, test_predictions = [], {}
for family in ['ls_known_only', 'ls_all_rows']:
    started = time.perf_counter()
    pipeline = fit_pipeline(family, 35)
    if family == 'ls_all_rows':
        assert pipeline['unknown_rows'] == int(train.label.isna().sum())
        assert pipeline['fitting_rows'] == len(train)
    probabilities, fallbacks = predict_pipeline(pipeline, test, audit_library=True)
    mapping = pd.Series(probabilities, index=test['index'])
    submission = template[['index']].copy()
    submission['target'] = submission['index'].map(mapping)
    assert submission.columns.tolist() == ['index', 'target']
    assert submission['index'].equals(template['index']) and submission.target.notna().all()
    submission_path = OUTPUT_DIR / f'submission_{family}_experimental.csv'
    model_path = OUTPUT_DIR / f'labelspreading_{family}_experimental.joblib'
    submission.to_csv(submission_path, index=False, float_format='%.17g')
    joblib.dump(pipeline, model_path, compress=3)
    restored = joblib.load(model_path)
    restored_probabilities, restored_fallbacks = predict_pipeline(restored, test)
    assert np.array_equal(restored_probabilities, probabilities)
    assert restored_fallbacks == fallbacks
    csv = pd.read_csv(submission_path)
    assert csv['index'].equals(template['index'])
    assert np.allclose(csv.target, submission.target, rtol=0, atol=1e-12)
    test_predictions[family] = probabilities
    final_records.append(dict(family=family, fitting_rows=pipeline['fitting_rows'],
                              known_rows=pipeline['known_rows'], unknown_rows=pipeline['unknown_rows'],
                              n_iter=pipeline['n_iter'], convergence_warning=pipeline['convergence_warning'],
                              zero_mass_fitted_rows=pipeline['zero_mass_fitted_rows'],
                              fallback_queries=fallbacks, test_rows=len(test),
                              seconds=round(time.perf_counter() - started, 3),
                              submission_name=submission_path.name, model_name=model_path.name))
    print('Full-data candidate:', family, '; fitted rows:', pipeline['fitting_rows'],
          '; unknown rows:', pipeline['unknown_rows'], '; fallback queries:', fallbacks,
          '; reloaded model and CSV checks passed.', flush=True)
    del pipeline, restored
    gc.collect()
np.savez_compressed(OUTPUT_DIR / 'test_probabilities.npz',
                    txIds=test.txId.to_numpy(), indices=test['index'].to_numpy(), **test_predictions)
pd.DataFrame(final_records).to_csv(OUTPUT_DIR / 'final_training.csv', index=False)
display(pd.DataFrame(final_records))


Full-data candidate: ls_known_only ; fitted rows: 31235 ; unknown rows: 0 ; fallback queries: 0 ; reloaded model and CSV checks passed.
Full-data candidate: ls_all_rows ; fitted rows: 141772 ; unknown rows: 110537 ; fallback queries: 8 ; reloaded model and CSV checks passed.


,family,fitting_rows,known_rows,unknown_rows,n_iter,convergence_warning,zero_mass_fitted_rows,fallback_queries,test_rows,seconds,submission_name,model_name
0,ls_known_only,31235,31235,0,10,False,0,0,15329,3.596,submission_ls_known_only_experimental.csv,labelspreading_ls_known_only_experimental.joblib
1,ls_all_rows,141772,31235,110537,10,False,10399,8,15329,38.617,submission_ls_all_rows_experimental.csv,labelspreading_ls_all_rows_experimental.joblib


## 6. Verify and disclose assistance

The student chose this fixed LabelSpreading mechanism, settings, all-row/known-only comparison and four chronological checks. OpenAI Codex implemented and ran that comparison, checked boundaries and artifacts, and drafted the factual record. The student must understand and defend the choices, interpret results, select the final submission and disclose this assistance accurately.

Original-control historical arrays, if available, are used only as an optional reproducibility audit; they are not needed to train or generate predictions from the supplied CSVs. No outer/test features or answers fit the graph or scaler. No automatic final-model selection, further material refinement or Kaggle upload occurs.

In [6]:
for end, first, last in CHECKS:
    expected = known.loc[known.time_step.between(first, last)]
    assert np.array_equal(saved[f'txIds_{end}'], expected.txId.to_numpy())
    assert np.array_equal(saved[f'labels_{end}'], expected.label.to_numpy())
    for family in FAMILIES:
        assert abs(roc_auc_score(saved[f'labels_{end}'], saved[f'{family}_{end}'])
                   - comparison.loc[end, family]) < 1e-12
previous_path = Path.cwd() / 'feature_ensemble_results' / 'validation_probabilities.npz'
historical_controls_checked = previous_path.is_file()
if historical_controls_checked:
    with np.load(previous_path) as previous:
        for end, first, last in CHECKS:
            assert np.array_equal(saved[f'labels_{end}'], previous[f'labels_{end}'])
            assert np.array_equal(saved[f'original_xgboost_{end}'], previous[f'original_xgboost_{end}'])

def file_hash(path):
    hasher = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            hasher.update(chunk)
    return hasher.hexdigest()

metadata = dict(
    date='2026-10-11', student_decision='train the fixed direct LabelSpreading comparison',
    method='feature similarity graph label spreading; inductive nearest-training-neighbour queries',
    labelspreading_config=LS_PARAMETERS, features=FEATURES,
    preprocessing='StandardScaler fit on respective fitting rows; clip [-10,10]; float32 inputs',
    cpu_threads=CPU_THREADS, distance_working_memory_mib=DISTANCE_MEMORY_MIB,
    query_batch=QUERY_BATCH, graph_storage='sparse',
    input_rows=len(train), known_labels=len(known), unknown_rows=int(train.label.isna().sum()),
    initial_unknown_label=-1, inferred_distributions_are_not_ground_truth=True,
    zero_information_policy='known-fitting illicit share if query neighbour label mass is zero',
    checks=CHECKS, validation_training=records, final_training=final_records,
    summary=summary.to_dict(orient='index'),
    all_rows_mean_change=float(comparison.all_rows_minus_original.mean()),
    all_rows_mean_change_vs_known_only=float(comparison.all_rows_minus_known_only.mean()),
    original_controls_exactly_reproduced=historical_controls_checked,
    all_saved_components_reload_exactly=True,
    library_prediction_equivalence_checked_on_informed_query_sample=True,
    no_outer_features_or_labels_in_fitting=True, no_test_features_in_fitting=True,
    outer_checks_overlap=True, outer_checks_are_development_evidence=True,
    original_public_auc=0.94265, candidate_public_auc=None,
    student_final_selection='pending review', kaggle_submission_made=False,
    python=platform.python_version(), sklearn=sklearn.__version__, xgboost=xgboost.__version__,
    pandas=pd.__version__, numpy=np.__version__, joblib=joblib.__version__,
    elapsed_seconds=round(time.perf_counter() - started_all, 3),
    input_sha256={name:file_hash(DATA_DIR/name) for name in [
        'Train Data.csv', 'Test Data.csv', 'Sample Submission/BT4012 Competition Submission.csv']},
    submission_sha256={family:file_hash(OUTPUT_DIR/f'submission_{family}_experimental.csv')
                       for family in ['ls_known_only','ls_all_rows']},
)
(OUTPUT_DIR / 'experiment_metadata.json').write_text(json.dumps(metadata, indent=2))
progress(phase='complete', all_rows_mean_auc=float(summary.loc['ls_all_rows','mean']),
         original_mean_auc=float(summary.loc['original_xgboost','mean']))
print('Chronology, probability alignment, AUC, historical controls, reloaded components and CSV checks passed.')
print('No Kaggle upload or automatic final-model selection was made.')


Chronology, probability alignment, AUC, historical controls, reloaded components and CSV checks passed.
No Kaggle upload or automatic final-model selection was made.


## 7. Measured result and interpretation

All six code cells executed in sequence and passed. The original-control probabilities exactly reproduced the earlier experiment. Independent AUC recalculation and identifier/count checks passed. Saved model/scaler bundles reproduced identical full test predictions after reload. The selected settings were not changed after seeing the outer scores.

| Checking steps | Original XGBoost | Known-only LabelSpreading | All-row LabelSpreading | All-row change vs original |
|---|---:|---:|---:|---:|
| 15–28 | 0.968521 | 0.921993 | 0.903163 | -0.065359 |
| 18–31 | 0.925611 | 0.861706 | 0.874425 | -0.051187 |
| 20–33 | 0.901582 | 0.850331 | 0.828829 | -0.072753 |
| 22–35 | 0.915330 | 0.854200 | 0.834371 | -0.080959 |
| **Mean** | **0.927761** | **0.872057** | **0.860197** | **-0.067565** |

The all-row condition improved **0 of four** checks against original XGBoost, with mean change **-0.067565 AUC**. Adding unknown rows to LabelSpreading improved **1 of four** checks against known-only LabelSpreading, with mean change **-0.011861**. Its worst fourteen-period AUC was **0.828829**. These measurements assess this specific feature-similarity graph and prediction rule; they do not show that all semi-supervised learning methods fail.

**AI-assisted recommendation:** Do not spend a remaining submission on this all-row LabelSpreading candidate; it declined against original XGBoost on every chronological check. The original XGBoost remains the best recorded public result, **0.94265**. The new candidate's public/private scores are unknown. Local chronological AUC and public leaderboard AUC are different evaluations and should not be treated as interchangeable.

### Unknown rows and information coverage

| Cutoff | All-row fitting transactions | Unknown fitting transactions | Propagation iterations | Zero-information fitting nodes | Fallback checking queries |
|---|---:|---:|---:|---:|---:|
| 14 | 70,990 | 56,094 | 9 | 6,973 | 5 |
| 17 | 80,989 | 64,134 | 9 | 8,460 | 17 |
| 19 | 86,471 | 68,482 | 9 | 9,219 | 5 |
| 21 | 94,299 | 74,769 | 10 | 8,260 | 4 |

All selected fits converged within the fixed iteration budget: **0 convergence warnings** in the eight LabelSpreading validation fits. A zero-information fitting node has no nonzero class distribution after propagation. It is not evidence that the transaction is licit. If a query's 15 neighbours collectively have zero distribution mass, the selected training-prior fallback is used and counted.

The full all-row graph and scaler used **141,772 training transactions**, including **110,537 unknown transactions**. Confirmed labels alone initialized class information. The full all-row fit used **10 iterations**, left **10,399 zero-information fitted nodes, and used the fallback for **8 of the **15,329** test queries. Both experimental CSVs passed alignment and finite-score checks. Test rows were queried after fitting; they did not train the scaler or graph.

Feature similarity is the hypothesis being tested. The results do not establish that the standardized 165-feature distances identify the same relationships as fraud risk, or diagnose the cause of every decline. With the selected inductive rule, future queries cannot relay information through one another. Introducing future feature nodes or changing the distance rule would be a new material experiment, not part of this run.

### Artifacts and decision ownership

Complete source and genuine outputs are in `e1156285_labelspreading.ipynb`. Measured tables, checking probabilities, metadata, full fitted bundles and experimental CSVs are in `labelspreading_results/`. Prediction generation requires only the notebook, supplied CSVs and recorded libraries; prior project prediction arrays are optional audit inputs.

The student explicitly selected the LabelSpreading mechanism, fixed settings and validation design. OpenAI Codex implemented and ran that comparison, verified the library prediction calculation, checked results and artifacts, and drafted this factual record. The student retains responsibility for interpretation, future material choices and final submission selection, and must accurately disclose assistance. The four overlapping checks are reused development evidence, not independent held-out proof.

No Kaggle submission, automatic further tuning or final-model selection was made. No improvement over the recorded public high score or guarantee of 0.96+ has been established.
